<a href="https://colab.research.google.com/github/rajivagarwal221/Aahar-Kitchen-System/blob/main/notebooks/combine_aahar_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AHAAR — combine the three detectors into one file

Packs the **ingredient** (107 classes), **dish** (20) and **packaged-brand** (59) YOLOv11m models into a single `aahar_combined.onnx` that runs all three in one forward pass and returns one list over 186 classes. Nothing is retrained, so accuracy is unchanged; cell 5 checks that on a photo.

The backend picks the file up automatically from Hugging Face (`Rajiv221/aahar-combined-detector`) on its next restart.

**Before Run all**
1. A GPU is **not** needed (CPU runtime is fine, ~5 min).
2. Colab secret `HF_TOKEN` (the Write token you already made).
3. The dish model's `best.pt` somewhere in Google Drive. Cell 4 searches `MyDrive/AAhar` for it; or set `DISH_WEIGHTS` in cell 2.

In [ ]:
!pip install -q "ultralytics==8.3.*" onnx onnxruntime onnxslim "huggingface_hub==0.26.*" 2>&1 | tail -2

## 2. Config

In [ ]:
REPO_URL    = 'https://github.com/rajivagarwal221/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'

HF_INGREDIENTS = 'Rajiv221/aahar-ingredient-detector'   # best.pt
HF_PACKAGED    = 'Rajiv221/aahar-packaged-detector'     # best.pt
HF_COMBINED    = 'Rajiv221/aahar-combined-detector'     # where the combined file goes

DRIVE_ROOT   = '/content/drive/MyDrive/AAhar'
DISH_WEIGHTS = ''        # e.g. '/content/drive/MyDrive/AAhar/runs/yolov11m_if20pretrain_stage1/weights/best.pt'; '' = search Drive
OUT          = '/content/aahar_combined.onnx'
UPLOAD_TO_HF = True
print('Config OK.')

## 3. Clone repo, mount Drive

In [ ]:
import os, subprocess, sys
from pathlib import Path
if not Path('/content/aahar').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, '/content/aahar'], check=True)
sys.path.insert(0, '/content/aahar/training'); sys.path.insert(0, '/content/aahar/backend')
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
import slots as S
print('expected classes:', {s: len(S.expected_classes(s) or []) for s in S.SLOTS})

## 4. Get the three models' weights

Ingredient and packaged models come from Hugging Face. The dish model is found in Drive by its class list (the 20 IndianFood20 classes).

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
token = userdata.get('HF_TOKEN')

weights = {
    'ingredients': hf_hub_download(HF_INGREDIENTS, 'best.pt', token=token),
    'packaged':    hf_hub_download(HF_PACKAGED, 'best.pt', token=token),
}

def names_of(pt):
    n = YOLO(pt).names
    return [n[i] for i in sorted(n)] if isinstance(n, dict) else list(n)

dish_expected = S.expected_classes('dishes')
if DISH_WEIGHTS:
    weights['dishes'] = DISH_WEIGHTS
else:
    found = []
    for pt in sorted(Path(DRIVE_ROOT).rglob('best.pt')):
        try:
            if names_of(str(pt)) == dish_expected:
                found.append(pt)
        except Exception:
            pass
    if found:
        weights['dishes'] = str(max(found, key=lambda p: p.stat().st_mtime))   # newest
    else:
        print(f'⚠️  No dish model with the 20 IndianFood20 classes found under {DRIVE_ROOT}.')
        print('    Set DISH_WEIGHTS in cell 2 and re-run, or continue to combine only ingredients + packaged.')

for slot, pt in weights.items():
    got, exp = names_of(pt), S.expected_classes(slot)
    print(f"{slot:12s} {len(got):3d} classes  {'matches repo list' if got == exp else 'DOES NOT match repo list'}  {pt}")

## 5. Combine and check

`verify` runs each original model and the combined file on the same photo and compares every detection. All rows should say **OK**.

In [ ]:
import combine_models as C, ultralytics, os
info = C.build(weights, Path(OUT))
sample = os.path.join(os.path.dirname(ultralytics.__file__), 'assets', 'bus.jpg')
print('\nAll match:', C.verify(weights, Path(OUT), sample))

### Optional: check on one of your own photos

In [ ]:
from google.colab import files
for name in files.upload():
    print(name, '→ all match:', C.verify(weights, Path(OUT), name))

## 6. Parameter count

In [ ]:
print(f"{'model':14s} {'classes':>7s} {'parameters':>12s}")
for slot, (lo, hi) in info['slots'].items():
    n = sum(p.numel() for p in C._prepare(weights[slot])[0].parameters())   # fused, as in the training logs
    print(f'{slot:14s} {hi - lo:7d} {n:12,d}')
print(f"{'combined':14s} {len(info['names']):7d} {info['params']:12,d}   ({Path(OUT).stat().st_size/1e6:.0f} MB)")

## 7. Save to Drive and upload to Hugging Face

In [ ]:
import shutil
shutil.copy(OUT, f'{DRIVE_ROOT}/aahar_combined.onnx'); print('Drive copy:', f'{DRIVE_ROOT}/aahar_combined.onnx')
if UPLOAD_TO_HF:
    from huggingface_hub import HfApi, create_repo
    create_repo(HF_COMBINED, token=token, exist_ok=True)
    HfApi(token=token).upload_file(path_or_fileobj=OUT, path_in_repo='aahar_combined.onnx', repo_id=HF_COMBINED)
    print(f'Uploaded → https://huggingface.co/{HF_COMBINED}')
    print('Restart the backend; /health should then show "mode": "combined".')